# StudySimilarityEngine Usage Examples

How to call the graph-backed similarity engine.

In [1]:
import sys
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.similarity import PublicStudySimilarityEngine, protocol_studies
import pandas as pd
names = protocol_studies()["study_name"].tolist()
engine = PublicStudySimilarityEngine()


## Example 1: Compare Two Studies

In [2]:

result = engine.compare(names[0], names[1])
print(result.study_pair, result.study_design_similarity)
display(result.component_breakdown)


('NCT02060721', 'NCT02099721') 0.28072792711430844


{'entity_type_similarity': {'raw_score': 0.0,
  'weight_applied': 0.3,
  'weighted_contribution': 0.0},
 'section_distribution_similarity': {'raw_score': 0.42023031949290335,
  'weight_applied': 0.25,
  'weighted_contribution': 0.10505757987322584},
 'entity_deviation_similarity': {'raw_score': 0.3083156103785641,
  'level1_score': 0.44045087196937727,
  'level2_score': 0.0,
  'weight_applied': 0.25,
  'weighted_contribution': 0.07707890259464102},
 'protocol_embedding_similarity': {'raw_score': 0.4906096309762772,
  'weight_applied': 0.15,
  'weighted_contribution': 0.07359144464644157},
 'metadata_similarity': {'raw_score': 0.5,
  'weight_applied': 0.05,
  'weighted_contribution': 0.025}}

## Example 2: Find Top N Similar Studies

In [3]:

top = engine.find_top_n_similar(names[0], n=5)
display(pd.DataFrame([{"study_id": item.study_pair[1], "similarity": item.study_design_similarity} for item in top]))


,study_id,similarity
0,NCT02427958,0.664822
1,NCT02879578,0.645054
2,NCT02709655,0.618838
3,NCT02188121,0.616638
4,NCT03461016,0.573085


## Example 3: Access Data Artifacts

In [4]:

display(pd.DataFrame(result.artifacts["entity_comparison"]).query("status != 'absent'").head(10))
display(pd.DataFrame(result.artifacts["metadata_comparison"]))


,entity,count_a,count_b,status
4,ASSESSMENT- 6MWT,1,0,unique_to_a
13,ASSESSMENT- Approval of the CIP,0,1,unique_to_b
30,ASSESSMENT- Borg dyspnea index,1,0,unique_to_a
39,ASSESSMENT- Change in vital signs and body weight,1,0,unique_to_a
46,ASSESSMENT- Clinical trial registration,0,1,unique_to_b
66,ASSESSMENT- EF less than 35%,0,1,unique_to_b
69,ASSESSMENT- Ethics Committee approval,0,1,unique_to_b
75,ASSESSMENT- Final study contact,0,1,unique_to_b
77,ASSESSMENT- Frequency of PVCs,0,1,unique_to_b
92,ASSESSMENT- Identify high-risk primary prevent...,0,1,unique_to_b


,field,a,b,included,match
0,study_phase,PHASE2,NA,True,False
1,indication,Chronic Thromboembolic Pulmonary Hypertension,Sudden Cardiac Arrest,True,False
2,subject_type,PATIENTS,PATIENTS,True,True
3,design,SINGLE_GROUP,PARALLEL,True,False
4,blinding,NONE,NONE,True,True
5,comparative_study,N,N,True,True
6,allocation,NA,NON_RANDOMIZED,True,False
7,sex,ALL,ALL,True,True


## Example 4: Custom Pipeline Parameters

In [5]:

custom = engine.compare(names[0], names[2], weights={"entity_type": 0.5, "section_distribution": 0.2, "entity_deviation": 0.1, "protocol_embedding": 0.1, "metadata": 0.1})
print(custom.study_design_similarity, custom.weights)


0.6487868619153935 {'entity_type': 0.5, 'section_distribution': 0.2, 'entity_deviation': 0.1, 'protocol_embedding': 0.1, 'metadata': 0.1}


## Example 5: Using LLM Reviewer for Mapping Corrections

The reviewer flag is off in this example. Turn it on after pointing `LLM_BASE_URL` at a local model.

In [6]:

reviewed = engine.compare(names[0], names[1], use_llm_reviewer=False)
print("reviewer_applied", False, "score", reviewed.study_design_similarity)


reviewer_applied False score 0.28072792711430844


## Example 6: Comparing with Specific Candidate Studies

In [7]:

subset = engine.find_top_n_similar(names[0], n=3, candidates=names[1:6])
display(pd.DataFrame([{"study_id": item.study_pair[1], "similarity": item.study_design_similarity} for item in subset]))


,study_id,similarity
0,NCT02427958,0.664822
1,NCT02188121,0.616638
2,NCT02374255,0.517886
